# 13 · Generate Parentheses & Combination Sum (I, II, III)

Backtracking problems where the recursion has **rules**: balance of brackets, limited/unlimited reuse of numbers, and fixed-size combinations.

**Topics covered**
1. Generate valid parentheses
2. Combination Sum I: numbers may be reused unlimited times
3. Combination Sum II: each number used at most once, input has duplicates
4. Combination Sum III: pick exactly `k` distinct digits from 1-9 that add up to `n`

*(The "Subsets II" solution that was also in this file lives in notebook 12.)*

## 1. Generate parentheses

Given `n` pairs, produce every **well-formed** string of `n` `(` and `n` `)`.
For `n = 3`: `((()))`, `(()())`, `(())()`, `()(())`, `()()()` → 5 strings (Catalan numbers: 1, 2, 5, 14, 42, …).

**Rules while building the string**
- You may add `(` as long as `open < n`.
- You may add `)` only if `close < open` (there is an unmatched `(` to close).

Those two rules guarantee we never create an invalid prefix, so no repair or filtering is needed.

**Time:** O(4ⁿ / √n) · **Space:** O(n)

In [1]:
def generate_parentheses(n):
    result = []
    current = []

    def build(open_count, close_count):
        if len(current) == 2 * n:                    # string complete
            result.append("".join(current))
            return
        if open_count < n:                           # still have "(" available
            current.append("(")
            build(open_count + 1, close_count)
            current.pop()                            # backtrack
        if close_count < open_count:                 # a ")" would match an open "("
            current.append(")")
            build(open_count, close_count + 1)
            current.pop()                            # backtrack

    build(0, 0)
    return result

print(generate_parentheses(2))                       # ['(())', '()()']
print(generate_parentheses(3))
print([len(generate_parentheses(n)) for n in range(1, 8)])   # Catalan: 1, 2, 5, 14, 42, 132, 429

['(())', '()()']
['((()))', '(()())', '(())()', '()(())', '()()()']
[1, 2, 5, 14, 42, 132, 429]


### The "balance counter" version
Instead of two counters, keep one `total` = (opens − closes) while filling `2n` slots. A prefix is valid while `0 ≤ total ≤ n`, and the string is valid when it ends at `total == 0`.

In [2]:
def generate_parentheses_balance(n):
    result = []
    bracket = [" "] * (2 * n)

    def solve(index, total):
        if total < 0 or total > n:                   # too many ")" or too many "(" can still be closed
            return
        if index == 2 * n:
            if total == 0:
                result.append("".join(bracket))
            return
        bracket[index] = "("
        solve(index + 1, total + 1)
        bracket[index] = ")"
        solve(index + 1, total - 1)

    solve(0, 0)
    return result

assert sorted(generate_parentheses_balance(4)) == sorted(generate_parentheses(4))
print(generate_parentheses_balance(2))

['(())', '()()']


## 2. Combination Sum I: unlimited reuse

Find all **unique combinations** of `candidates` whose sum equals `target`. The same number may be used **any number of times**. All candidates are distinct and positive.

**Pick / not-pick with a twist**
- **Pick:** add `candidates[index]` and **stay at the same index** (we may pick it again).
- **Not pick:** move to `index + 1`.

```
candidates = [2, 4, 5, 6], target = 9
[2, 2, 5] ✔   [4, 5] ✔
```

Stop when `total == target` (record) or `total > target` (prune) or `index` is out of range.

**Time:** about O(2^t) where `t = target / min(candidates)` · **Space:** O(t)

In [3]:
class CombinationSum:
    def solve(self, index, total, subset, nums, target, result):
        if total == target:
            result.append(subset.copy())
            return
        if total > target or index >= len(nums):
            return
        subset.append(nums[index])                                                # PICK (stay on index)
        self.solve(index, total + nums[index], subset, nums, target, result)
        subset.pop()                                                              # backtrack
        self.solve(index + 1, total, subset, nums, target, result)                # NOT PICK (move on)

    def combination_sum(self, candidates, target):
        result = []
        self.solve(0, 0, [], candidates, target, result)
        return result

print(CombinationSum().combination_sum([2, 4, 5, 6], 9))        # [[2, 2, 5], [4, 5]]
print(CombinationSum().combination_sum([2, 3, 6, 7], 7))        # [[2, 2, 3], [7]]

[[2, 2, 5], [4, 5]]
[[2, 2, 3], [7]]


## 3. Combination Sum II: each number once, with duplicates

`candidates = [10, 1, 2, 7, 6, 1, 5]`, `target = 8` → `[1,1,6] [1,2,5] [1,7] [2,6]`.
Every element can be used **at most once**, and the answer must contain **no duplicate combinations**.

Combine two ideas you already know:
1. **Sort**, then in the `for` loop skip a value equal to the previous one at the same level: `if i > index and nums[i] == nums[i-1]: continue` (from Subsets II).
2. Recurse with **`i + 1`** because an element may not be reused.

Extra pruning: because the array is sorted, once `nums[i] > remaining` every later number is too big as well, so `break`.

> 🐞 **Fix:** the earlier cell was only a fragment (it used the undefined variables `nums`, `n` and `result`). Here it's a complete, runnable class.

In [4]:
class CombinationSum2:
    def solve(self, index, remaining, subset, nums, result):
        if remaining == 0:                          # exact match
            result.append(subset.copy())
            return
        for i in range(index, len(nums)):
            if i > index and nums[i] == nums[i - 1]:     # skip duplicate at the same level
                continue
            if nums[i] > remaining:                      # sorted → everything after is too big too
                break
            subset.append(nums[i])
            self.solve(i + 1, remaining - nums[i], subset, nums, result)   # i + 1: no reuse
            subset.pop()

    def combination_sum2(self, candidates, target):
        result = []
        self.solve(0, target, [], sorted(candidates), result)
        return result

print(CombinationSum2().combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))   # [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
print(CombinationSum2().combination_sum2([2, 5, 2, 1, 2], 5))          # [[1, 2, 2], [5]]

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
[[1, 2, 2], [5]]


## 4. Combination Sum III

Use **exactly `k` numbers**, each from `1 … 9` and each **used at most once**, so that they add up to `n`.
Example: `k = 3, n = 9` → `[1,2,6] [1,3,5] [2,3,4]`.

Pick / not-pick over the digits `1 … 9`, with three stopping rules:
- `len(subset) == k and total == n` → record
- `len(subset) > k` or `total > n` → prune
- index past the digit `9` → stop

In [5]:
class CombinationSum3:
    def solve(self, nums, index, total, subset, k, n, result):
        if total == n and len(subset) == k:
            result.append(subset.copy())
            return
        if len(subset) >= k or total > n or index >= len(nums):    # can't succeed on this branch
            return
        subset.append(nums[index])                                  # pick
        self.solve(nums, index + 1, total + nums[index], subset, k, n, result)
        subset.pop()                                                # backtrack
        self.solve(nums, index + 1, total, subset, k, n, result)    # not pick

    def combination_sum3(self, k, n):
        result = []
        self.solve([1, 2, 3, 4, 5, 6, 7, 8, 9], 0, 0, [], k, n, result)
        return result

print(CombinationSum3().combination_sum3(3, 7))      # [[1, 2, 4]]
print(CombinationSum3().combination_sum3(3, 9))      # [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
print(CombinationSum3().combination_sum3(4, 1))      # []  (impossible)

# Cross-check against itertools
from itertools import combinations
for k, n in [(3, 7), (3, 9), (4, 20), (2, 17)]:
    expected = sorted(list(c) for c in combinations(range(1, 10), k) if sum(c) == n)
    assert sorted(CombinationSum3().combination_sum3(k, n)) == expected
print("✅ matches itertools")

[[1, 2, 4]]
[[1, 2, 6], [1, 3, 5], [2, 3, 4]]
[]
✅ matches itertools


---
### ✅ Key takeaways
- **Parentheses:** add `(` while `open < n`; add `)` only while `close < open`.
- **Reuse allowed** → recurse with the *same* index after picking. **No reuse** → `i + 1`.
- **Duplicates in input** → sort, then `if i > index and nums[i] == nums[i-1]: continue`.
- A sorted array lets you `break` early once the current number exceeds the remaining target.